In [8]:
%pip install tree-sitter tree-sitter-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Sample Scanner Output Setup

In [9]:
import json

# Step 1: Create a sample raw scan output coming from security scanner
raw_scanner_data = {
    "results": [
        {
            "rule_id": "sql-injection-risk",
            "file_path": "src/controllers/auth.py",
            "line_number": 42,
            "severity": "HIGH",
            "extra": {
                "message": "Potential SQL injection in user query",
                "function_name": "handle_login"
            }
        },
        {
            "rule_id": "hardcoded-secret-key",
            "file_path": "src/utils/test_helpers.py",
            "line_number": 105,
            "severity": "MEDIUM",
            "extra": {
                "message": "Found hardcoded API key in test script",
                "function_name": "deprecated_mock_auth"
            }
        },
        {
            "rule_id": "xss-vulnerability",
            "file_path": "src/controllers/user.py",
            "line_number": 88,
            "severity": "HIGH",
            "extra": {
                "message": "Unsanitized user input rendered in response",
                "function_name": "get_user_profile"
            }
        },
        {
            "rule_id": "command-injection",
            "file_path": "src/utils/old_calc.py",
            "line_number": 15,
            "severity": "HIGH",
            "extra": {
                "message": "Unchecked system call execution",
                "function_name": "unused_calculator_stub"
            }
        }
    ]
}

# Saving sample data into a local JSON file for testing
with open("raw_scan_results.json", "w") as file:
    json.dump(raw_scanner_data, file, indent=4)

print("Setup Complete: Created 'raw_scan_results.json' file successfully!")

Setup Complete: Created 'raw_scan_results.json' file successfully!


## AST Noise Filter Engine (Real + Tree-Sitter PoC)

In [10]:
import json
import tree_sitter_python as tspython
from tree_sitter import Language, Parser

class ASTNoiseFilter:
    """
    Analyzes AST reachability to separate actionable alerts from dead code noise.
    """
    def __init__(self, main_entry_points):
        self.entry_points = main_entry_points
        # Tree-sitter parser setup
        self.PY_LANGUAGE = Language(tspython.language())
        self.parser = Parser(self.PY_LANGUAGE)

    def parse_code_structure(self, code_snippet):
        """
        Parses source code into AST Syntax Tree using tree-sitter.
        """
        tree = self.parser.parse(bytes(code_snippet, "utf8"))
        return tree.root_node

    def get_reachable_functions(self):
        # Simulated call graph mapping reachable function execution paths
        active_call_graph = {
            "handle_login": ["validate_token", "fetch_user_db"],
            "get_user_profile": ["fetch_user_db", "render_template"]
        }
        
        reachable_set = set(self.entry_points)
        for parent, children in active_call_graph.items():
            if parent in reachable_set:
                reachable_set.update(children)
                
        return reachable_set

    def process_scan_file(self, json_file_path):
        with open(json_file_path, "r") as file:
            scan_payload = json.load(file)
            
        alerts = scan_payload.get("results", [])
        reachable_nodes = self.get_reachable_functions()
        
        valid_alerts = []
        filtered_noise = []

        for alert in alerts:
            target_func = alert["extra"].get("function_name", "unknown")
            
            if target_func in reachable_nodes:
                alert["status"] = "REACHABLE_CODE"
                alert["action_required"] = True
                valid_alerts.append(alert)
            else:
                alert["status"] = "UNREACHABLE_DEAD_CODE"
                alert["action_required"] = False
                filtered_noise.append(alert)

        return valid_alerts, filtered_noise

print("ASTNoiseFilter Class with tree-sitter integrated successfully!")

ASTNoiseFilter Class with tree-sitter integrated successfully!


## Live Output Execution

In [11]:
# Initialize AST filter with primary API entry points
filter_engine = ASTNoiseFilter(main_entry_points=["handle_login", "get_user_profile"])

# Test tree-sitter AST parsing on a sample code string
sample_code = "def handle_login(): pass"
ast_root = filter_engine.parse_code_structure(sample_code)
print(f"Tree-sitter AST Root Node Verified: {ast_root.type}\n")

# Execute reachability analysis on raw scan file
valid_findings, noise_findings = filter_engine.process_scan_file("raw_scan_results.json")

# Print summary dashboard metrics
print("=" * 65)
print("     AUTOFORGE AST NOISE FILTER - DEMO RESULT SUMMARY")
print("=" * 65)
print(f"Total Security Alerts (Raw Input)     : {len(valid_findings) + len(noise_findings)}")
print(f"Filtered Out Alerts (Dead/Unused Code) : {len(noise_findings)}")
print(f"Actionable Security Findings          : {len(valid_findings)}")
print("=" * 65)

print("\n[ACTIVE ALERTS -> SENT TO DASHBOARD]")
for item in valid_findings:
    print(f" • Rule ID   : {item['rule_id']}")
    print(f"   Location  : {item['file_path']} (Line {item['line_number']})")
    print(f"   Function  : {item['extra']['function_name']}()")
    print(f"   Status    : {item['status']}\n")

print("[ISOLATED NOISE -> DEAD CODE IGNORED]")
for item in noise_findings:
    print(f" • Rule ID   : {item['rule_id']}")
    print(f"   Location  : {item['file_path']} (Line {item['line_number']})")
    print(f"   Function  : {item['extra']['function_name']}()")
    print(f"   Status    : {item['status']}\n")

Tree-sitter AST Root Node Verified: module

     AUTOFORGE AST NOISE FILTER - DEMO RESULT SUMMARY
Total Security Alerts (Raw Input)     : 4
Filtered Out Alerts (Dead/Unused Code) : 2
Actionable Security Findings          : 2

[ACTIVE ALERTS -> SENT TO DASHBOARD]
 • Rule ID   : sql-injection-risk
   Location  : src/controllers/auth.py (Line 42)
   Function  : handle_login()
   Status    : REACHABLE_CODE

 • Rule ID   : xss-vulnerability
   Location  : src/controllers/user.py (Line 88)
   Function  : get_user_profile()
   Status    : REACHABLE_CODE

[ISOLATED NOISE -> DEAD CODE IGNORED]
 • Rule ID   : hardcoded-secret-key
   Location  : src/utils/test_helpers.py (Line 105)
   Function  : deprecated_mock_auth()
   Status    : UNREACHABLE_DEAD_CODE

 • Rule ID   : command-injection
   Location  : src/utils/old_calc.py (Line 15)
   Function  : unused_calculator_stub()
   Status    : UNREACHABLE_DEAD_CODE



## Convert notebook logic directly into pipeline script

In [12]:
script_content = '''import json
import tree_sitter_python as tspython
from tree_sitter import Language, Parser

class ASTNoiseFilter:
    def __init__(self, main_entry_points):
        self.entry_points = main_entry_points
        self.PY_LANGUAGE = Language(tspython.language())
        self.parser = Parser(self.PY_LANGUAGE)

    def process_scan_file(self, json_file_path):
        with open(json_file_path, "r") as file:
            scan_payload = json.load(file)
            
        alerts = scan_payload.get("results", [])
        active_call_graph = {
            "handle_login": ["validate_token", "fetch_user_db"],
            "get_user_profile": ["fetch_user_db", "render_template"]
        }
        
        reachable_nodes = set(self.entry_points)
        for parent, children in active_call_graph.items():
            if parent in reachable_nodes:
                reachable_nodes.update(children)
                
        valid_alerts = []
        filtered_noise = []

        for alert in alerts:
            target_func = alert["extra"].get("function_name", "unknown")
            if target_func in reachable_nodes:
                alert["status"] = "REACHABLE_CODE"
                valid_alerts.append(alert)
            else:
                alert["status"] = "UNREACHABLE_DEAD_CODE"
                filtered_noise.append(alert)

        return valid_alerts, filtered_noise

if __name__ == "__main__":
    filter_engine = ASTNoiseFilter(main_entry_points=["handle_login", "get_user_profile"])
    valid, noise = filter_engine.process_scan_file("raw_scan_results.json")
    print(f"Filter Complete: {len(valid)} Valid Alerts, {len(noise)} Noise Alerts Isolated.")
'''

with open("noise_filter.py", "w") as f:
    f.write(script_content)

print(" 'noise_filter.py' done")

 'noise_filter.py' done
